In [10]:
import sys, gc, os
sys.path.append("..")
import pandas as pd
from src.data_loader import download_and_load_brfss

shortlist = ["_MICHD", "_AGE_G", "SEXVAR", "_RACE", "EDUCA", "INCOME3",
             "GENHLTH", "_BMI5", "DIABETE4", "_RFHYPE6", "TOLDHI3",
             "_SMOKER3", "_TOTINDA", "SLEPTIM1", "_RFDRHV8"]

try:
    df_full = df
    assert df_full.shape[1] > 100
    print("Reusing the dataframe already in memory")
except (NameError, AssertionError):
    print("Downloading from the CDC (a minute or two)...")
    df_full = download_and_load_brfss()

print(df_full.shape)    # expect (457670, 301)
df.head()

(457670, 301)


,_MICHD,_AGE_G,SEXVAR,_RACE,EDUCA,INCOME3,GENHLTH,_BMI5,DIABETE4,_SMOKER3,_TOTINDA
0,2.0,6.0,2.0,1.0,4.0,99.0,3.0,2249.0,3.0,4.0,1.0
1,1.0,6.0,1.0,1.0,6.0,11.0,1.0,2583.0,3.0,3.0,1.0
2,2.0,5.0,1.0,1.0,5.0,99.0,2.0,2253.0,3.0,1.0,1.0
3,2.0,6.0,1.0,1.0,6.0,6.0,1.0,2509.0,3.0,4.0,1.0
4,2.0,4.0,1.0,1.0,5.0,3.0,3.0,1977.0,3.0,4.0,2.0


In [11]:
missing = [c for c in shortlist if c not in df_full.columns]
print("Missing columns:", missing)     # hopefully []

df = df_full[[c for c in shortlist if c in df_full.columns]].copy()
del df_full
gc.collect()

print(df.shape)                         # expect (457670, 15) if nothing is missing
df.head()

Missing columns: ['_RFHYPE6', 'TOLDHI3', 'SLEPTIM1', '_RFDRHV8']
(457670, 11)


,_MICHD,_AGE_G,SEXVAR,_RACE,EDUCA,INCOME3,GENHLTH,_BMI5,DIABETE4,_SMOKER3,_TOTINDA
0,2.0,6.0,2.0,1.0,4.0,99.0,3.0,2249.0,3.0,4.0,1.0
1,1.0,6.0,1.0,1.0,6.0,11.0,1.0,2583.0,3.0,3.0,1.0
2,2.0,5.0,1.0,1.0,5.0,99.0,2.0,2253.0,3.0,1.0,1.0
3,2.0,6.0,1.0,1.0,6.0,6.0,1.0,2509.0,3.0,4.0,1.0
4,2.0,4.0,1.0,1.0,5.0,3.0,3.0,1977.0,3.0,4.0,2.0


In [14]:
import os, gc

shortlist = [
    "_MICHD",
    # core
    "_AGE_G", "SEXVAR", "_RACE", "EDUCA", "INCOME3", "GENHLTH", "_BMI5",
    "DIABETE4", "_SMOKER3", "_TOTINDA", "_RFDRHV9",
    # chronic conditions
    "CHCKDNY2", "CHCCOPD3", "ASTHMA3", "ADDEPEV3", "CHCSCNC1", "CHCOCNC1", "HAVARTH4",
    # health and function
    "PHYSHLTH", "MENTHLTH", "DIFFWALK", "DIFFDRES", "DECIDE",
    # lifestyle
    "DRNKANY6", "_DRNKWK3",
    # social and access
    "MARITAL", "EMPLOY1", "MEDCOST1", "CHECKUP1",
]

missing = [c for c in shortlist if c not in df_full.columns]
print("Missing:", missing)           # must be []
assert not missing

df = df_full[shortlist].copy()
del df_full
gc.collect()

print(df.shape)                       # expect (457670, 30)

os.makedirs("../data/raw", exist_ok=True)
df.to_parquet("../data/raw/brfss2024_subset.parquet", index=False)
print(f"Saved: {os.path.getsize('../data/raw/brfss2024_subset.parquet')/1e6:.1f} MB")

Missing: []
(457670, 30)
Saved: 5.0 MB


In [15]:
for f in ["../data/raw/full_2024.parquet", "../data/raw/codebook24.zip", "data/raw/codebook24.zip"]:
    if os.path.exists(f):
        os.remove(f)
        print("Deleted", f)

In [16]:
print(df["_MICHD"].value_counts(dropna=False))

_MICHD
2.0    410126
1.0     42338
NaN      5206
Name: count, dtype: int64


CLEAN THE DATA

In [17]:
import pandas as pd

df = pd.read_parquet("../data/raw/brfss2024_subset.parquet")

for col in df.columns:
    n = df[col].nunique(dropna=False)
    print(f"\n=== {col}  (unique values: {n}) ===")
    if n <= 15:
        print(df[col].value_counts(dropna=False).sort_index().to_string())
    else:
        print(df[col].describe().to_string())
        print("most common:", df[col].value_counts(dropna=False).head(6).to_dict())


=== _MICHD  (unique values: 3) ===
_MICHD
1.0     42338
2.0    410126
NaN      5206

=== _AGE_G  (unique values: 6) ===
_AGE_G
1.0     29694
2.0     49994
3.0     60418
4.0     64458
5.0     80280
6.0    172826

=== SEXVAR  (unique values: 2) ===
SEXVAR
1.0    217487
2.0    240183

=== _RACE  (unique values: 9) ===
_RACE
1.0    329346
2.0     35172
3.0      6460
4.0     12646
5.0      2074
6.0      3737
7.0     10486
8.0     48646
9.0      9103

=== EDUCA  (unique values: 8) ===
EDUCA
1.0       723
2.0      9226
3.0     17200
4.0    115877
5.0    120732
6.0    191549
9.0      2356
NaN         7

=== INCOME3  (unique values: 14) ===
INCOME3
1.0     10258
2.0     10542
3.0     13662
4.0     19841
5.0     41702
6.0     49686
7.0     60709
8.0     51700
9.0     55778
10.0    26675
11.0    29694
77.0    37113
99.0    41041
NaN      9269

=== GENHLTH  (unique values: 8) ===
GENHLTH
1.0     64217
2.0    145791
3.0    156229
4.0     67919
5.0     22204
7.0       915
9.0       390
NaN         

In [18]:
import numpy as np
import pandas as pd

df = pd.read_parquet("../data/raw/brfss2024_subset.parquet")
print("start:", df.shape)

# 1. Target: drop unknown rows, then 1 = has CHD/MI, 0 = does not
df = df[df["_MICHD"].notna()].copy()
df["_MICHD"] = (df["_MICHD"] == 1).astype(int)
print("after dropping unknown target:", df.shape)

# 2. Yes/No questions (1 = yes, 2 = no; 7, 9 and blank become NaN)
yes_no = ["CHCKDNY2", "CHCCOPD3", "ASTHMA3", "ADDEPEV3", "CHCSCNC1", "CHCOCNC1",
          "HAVARTH4", "DIFFWALK", "DIFFDRES", "DECIDE", "MEDCOST1", "DRNKANY6"]
for c in yes_no:
    df[c] = df[c].map({1.0: 1, 2.0: 0})

# 3. Calculated risk flags, recoded so 1 = risk
df["_TOTINDA"] = df["_TOTINDA"].map({1.0: 0, 2.0: 1})   # 1 = physically inactive
df["_RFDRHV9"] = df["_RFDRHV9"].map({1.0: 0, 2.0: 1})   # 1 = heavy drinker

# 4. Categorical columns: unknown/refused codes become NaN
unknown_codes = {
    "_RACE": [9], "EDUCA": [9], "INCOME3": [77, 99], "GENHLTH": [7, 9],
    "DIABETE4": [7, 9], "_SMOKER3": [9], "MARITAL": [9], "EMPLOY1": [9],
    "CHECKUP1": [7, 9],
}
for c, codes in unknown_codes.items():
    df[c] = df[c].replace(codes, np.nan)

# 5. Days of poor health: 88 = none
for c in ["PHYSHLTH", "MENTHLTH"]:
    df[c] = df[c].replace({88: 0, 77: np.nan, 99: np.nan})

# 6. BMI is stored as BMI x 100
df["_BMI5"] = df["_BMI5"] / 100

# 7. Drinks per week
dw = df["_DRNKWK3"].copy()
dw[dw == 99900] = np.nan
dw[dw < 1e-6] = 0
df["_DRNKWK3"] = dw / 100

print("cleaned:", df.shape)

start: (457670, 30)
after dropping unknown target: (452464, 30)
cleaned: (452464, 30)


In [19]:
print("CHD prevalence:", round(df["_MICHD"].mean(), 4))
print("\nMissing % (top 12):")
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(12))

print("\n", df[["_BMI5", "_DRNKWK3", "PHYSHLTH", "MENTHLTH"]].describe().round(2))
print("\nBMI extremes:", df["_BMI5"].quantile([0.001, 0.999]).round(1).tolist())

assert df[["PHYSHLTH", "MENTHLTH"]].max().max() <= 30
assert df["_DRNKWK3"].max() < 999

CHD prevalence: 0.0936

Missing % (top 12):
INCOME3     18.9
_RFDRHV9    10.1
_DRNKWK3    10.1
DRNKANY6     9.5
_BMI5        9.3
_SMOKER3     6.9
DECIDE       4.4
DIFFWALK     4.4
DIFFDRES     4.4
PHYSHLTH     2.3
_RACE        1.9
EMPLOY1      1.8
dtype: float64

            _BMI5   _DRNKWK3   PHYSHLTH   MENTHLTH
count  410281.00  406603.00  441918.00  444707.00
mean       28.55       2.80       4.57       4.40
std         6.58       8.94       8.89       8.34
min        12.00       0.00       0.00       0.00
25%        24.14       0.00       0.00       0.00
50%        27.44       0.23       0.00       0.00
75%        31.75       2.33       4.00       5.00
max        99.84     532.00      30.00      30.00

BMI extremes: [14.8, 66.4]


In [21]:
os.makedirs("../data/processed", exist_ok=True)
df.to_parquet("../data/processed/cleaned.parquet", index=False)